# RobBERT

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset

In [ ]:
corpus_dataset = load_dataset("clips/mteb-nl-vabb-ret", "corpus", split="corpus")
df_corpus = pd.DataFrame(corpus_dataset)
df_corpus = df_corpus.rename(columns={'_id': 'corpus_id', 'text': 'document_text'})
df_corpus.head()

In [ ]:
full_dataset = pd.read_csv('abstracts_full_augmented.csv')
full_dataset.head()

In [ ]:
# 1. Volledige pool van documenten inladen
all_documents = df_corpus['document_text'].tolist()

# 2. Laad beide query-sets in
synthetic_queries = full_dataset["synthetic_query"].tolist()
original_queries = full_dataset["original_query"].tolist()

In [ ]:
def process_embeddings(model_name="DTAI-KULeuven/robbert-2023-dutch-base"):
    print(f"⏳ Model {model_name} loading...")
    model = SentenceTransformer(model_name, device="cuda")

    max_length = model.get_max_seq_length() if model.get_max_seq_length() is not None else 512
    model.max_seq_length = max_length

    # Pas aan naar 512 als je op een A100 zit, of 32/64 voor een T4 GPU
    current_batch_size = 256
    print(f"🏎️ Batch size ingesteld op {current_batch_size}.")

    print(f"📝 Standaard Nederlands model gedetecteerd (geen prefixes nodig).")
    print(f"🚀 Computing {len(all_documents)} document-embeddings...")

    doc_embeddings = model.encode(
        all_documents, # Gewoon de pure documenten, zonder E5-poustpas!
        batch_size=current_batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    clean_model_name = model_name.replace("/", "_")
    file_name = f"{clean_model_name}_corpus_embeddings_abstracts.npy"
    print(f"💾 Bezig met opslaan van document-embeddings naar '{file_name}'...")
    np.save(file_name, doc_embeddings)
    print("✅ Document-embeddings succesvol opgeslagen op de harde schijf!\n")

    # We returnen het model zodat je dat buiten de functie kunt hergebruiken voor de queries!
    return model

In [ ]:
# Run de functie en bewaar het geladen model
model = process_embeddings('DTAI-KULeuven/robbert-2023-dutch-base')

In [ ]:
print(f"🔍 NIEUWE queries embedden...")
generated_q_embeddings = model.encode(synthetic_queries, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print(f"🔍 OUDE queries embedden...")
original_q_embeddings = model.encode(original_queries, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

In [ ]:
doc_embeddings = np.load('DTAI-KULeuven_robbert-2023-dutch-base_corpus_embeddings_abstracts.npy')

In [ ]:
doc_embeddings.shape

In [ ]:
# Mocht je de documenten later in een schone sessie willen inladen, heractiveer dan deze regel:
# doc_embeddings = np.load('google-bert_bert-base-multilingual-cased_corpus_embeddings.npy')

print(f"🧮 Gelijkenis-matrices berekenen...")

matrix_storage = {}
model_name = 'DTAI-KULeuven/robbert-2023-dutch-base'

matrix_storage[f"{model_name}_NEW"] = cosine_similarity(generated_q_embeddings, doc_embeddings)
matrix_storage[f"{model_name}_OLD"] = cosine_similarity(original_q_embeddings, doc_embeddings)

print(f"✅ Alles succesvol afgerond voor {model_name}!")

In [ ]:
# ==========================================
# 🔍 JOUW CONTROLERONDJE (SANITY CHECK)
# ==========================================
print("📋 MATRICES CONTROLEREN:")
print("------------------------------------------")

for sleutel, matrix in matrix_storage.items():
    print(f"🔹 Matrix: {sleutel}")

    # Check 1: De vorm (Shape) -> Moet exact (1000, 255524) zijn
    print(f"   - Vorm (Shape):          {matrix.shape}")
    if matrix.shape == (1000, 255524):
        print("     ✅ Vorm is PERFECT! (1000 queries x 255.524 documenten)")
    else:
        print("     ❌ WAARSCHUWING: De vorm klopt niet!")

    # Check 2: Datatype -> Moet float32 of float64 zijn
    print(f"   - Datatype:              {matrix.dtype}")

    # Check 3: Zijn er lege velden (NaNs)? -> Moet 0 zijn
    nan_count = np.isnan(matrix).sum()
    print(f"   - Aantal lege cellen:    {nan_count}")
    if nan_count == 0:
        print("     ✅ Geen corrupte of lege waarden gevonden.")
    else:
        print(f"     ❌ WAARSCHUWING: Er zitten {nan_count} lege waarden in!")

    # Check 4: Realistische waarden? -> Cosine similarity moet tussen -1 en 1 liggen (meestal tussen 0 en 1)
    print(f"   - Bereik van scores:     Min: {matrix.min():.4f} tot Max: {matrix.max():.4f}")
    print(f"   - Voorbeeld score (0,0): {matrix[0, 0]:.4f}")
    print("------------------------------------------")

In [ ]:
# Sla de twee matrices gecomprimeerd op in één bestand (.npz)
# Tip: Verwijs eventueel naar je Google Drive path ('/content/drive/MyDrive/...') zodat het bestand blijft bestaan!
bestandsnaam = "similarity_matrices_robbert_abstracts.npz"

print(f"💾 Matrices aan het opslaan naar {bestandsnaam}...")
np.savez_compressed(
    bestandsnaam,
    matrix_new=matrix_storage[f"{model_name}_NEW"],
    matrix_old=matrix_storage[f"{model_name}_OLD"]
)
print("✅ Matrices veilig opgeslagen! Je kunt dit notebook nu sluiten en je GPU uitzetten.")

In [ ]:
import numpy as np

# 2. Geef het juiste pad op naar je .npz bestand
# Pas dit pad aan naar de map waar je hem hebt opgeslagen, bijv: '/content/drive/MyDrive/Script_Embeddings/similarity_matrices_mbert_news.npz'
bestandsnaam = "similarity_matrices_robbert_abstracts.npz"

print("🔄 Matrices inladen uit Google Drive...")
data = np.load(bestandsnaam)

# Haal de matrices weer boven water
matrix_new = data['matrix_new']
matrix_old = data['matrix_old']

print(f"✅ Succesvol geladen!")
print(f"   Shape Matrix Nieuw (Synthetic): {matrix_new.shape}") # Moet (1000, 255524) zijn
print(f"   Shape Matrix Oud (Original):    {matrix_old.shape}") # Moet (1000, 255524) zijn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

# 1. Zorg dat je matrices correct zijn gelinkt aan de variabelen uit je ingeladen .npz
sim_matrix_new = matrix_new  # Komt uit data['matrix_new']
sim_matrix_old = matrix_old  # Komt uit data['matrix_old']

query_level_results = []

# We lopen regel voor regel door de pilot-dataset heen (elke regel is een query)
for idx, row in full_dataset.iterrows():
    # Bepaal wat het ECHTE relevante document is voor deze query
    true_doc_id = row['corpus_id']

    # Maak een 'ground truth' array van nullen even lang als de corpus pool
    # Het document dat relevant is krijgt een 1
    true_relevance_bool = (df_corpus['corpus_id'] == true_doc_id).values
    true_relevance = true_relevance_bool.astype(int).reshape(1, -1)

    # Controleer of het relevante document wel in de pool zit
    if np.sum(true_relevance) == 0:
        continue # Overslaan als het document om een of andere reden ontbreekt

    # Haal de similarity scores op voor deze specifieke query (idx)
    scores_new = sim_matrix_new[idx]
    scores_old = sim_matrix_old[idx]

    # --- 1. Bereken de NDCG@10 score (Jouw vertrouwde methode) ---
    ndcg_new = ndcg_score(true_relevance, scores_new.reshape(1, -1), k=10)
    ndcg_old = ndcg_score(true_relevance, scores_old.reshape(1, -1), k=10)

    # --- 2. Bereken de MRR@10 score (Nieuwe toevoeging) ---
    # Sorteer de indices van hoog naar laag op basis van de similarity scores
    ranked_indices_new = np.argsort(scores_new)[::-1][:10]
    ranked_indices_old = np.argsort(scores_old)[::-1][:10]

    # Vind de exacte index in df_corpus van het échte document
    true_index_in_corpus = np.where(true_relevance_bool)[0][0]

    # Bereken MRR voor de nieuwe query
    mrr_new = 0.0
    if true_index_in_corpus in ranked_indices_new:
        rank = np.where(ranked_indices_new == true_index_in_corpus)[0][0] + 1
        mrr_new = 1.0 / rank

    # Bereken MRR voor de oude query
    mrr_old = 0.0
    if true_index_in_corpus in ranked_indices_old:
        rank = np.where(ranked_indices_old == true_index_in_corpus)[0][0] + 1
        mrr_old = 1.0 / rank

    # Sla de resultaten van deze specifieke query op
    query_level_results.append({
        'query_index': idx,
        'original_query': row['original_query'],
        'synthetic_query': row['synthetic_query'],
        'ndcg_10_old': ndcg_old,
        'ndcg_10_new': ndcg_new,
        'mrr_10_old': mrr_old,
        'mrr_10_new': mrr_new
    })

# Zet om naar een overzichtelijk DataFrame
df_query_scores = pd.DataFrame(query_level_results)

# Toon de algemene gemiddeldes op je scherm ter controle
print("📊 GEMIDDELDE METRIEKEN OVER DE HELE DATASET:")
print("--------------------------------------------------")
print(f"Oude Queries (Original)  -> NDCG@10: {df_query_scores['ndcg_10_old'].mean():.4f} | MRR@10: {df_query_scores['mrr_10_old'].mean():.4f}")
print(f"Nieuwe Queries (Synthetic) -> NDCG@10: {df_query_scores['ndcg_10_new'].mean():.4f} | MRR@10: {df_query_scores['mrr_10_new'].mean():.4f}")
print("--------------------------------------------------\n")

display(df_query_scores.head(10))

In [ ]:
df_query_scores.to_csv('query_scores_robbert_abstracts.csv', index=False)

In [ ]:
from scipy import stats

def voer_alle_testen_uit(df, kolom_oud, kolom_nieuw, metriek_naam):
    print(f"==================================================")
    print(f"🎲 SIGNIFICANTIETESTEN VOOR {metriek_naam}")
    print(f"==================================================")

    scores_oud = df[kolom_oud].values
    scores_nieuw = df[kolom_nieuw].values

    # 1. Paired t-test
    t_stat, t_p = stats.ttest_rel(scores_oud, scores_nieuw)
    print(f"👉 Paired t-test:          p-waarde = {t_p:.6f}  (t-stat = {t_stat:.4f})")

    # 2. Wilcoxon signed-rank test
    wilc_stat, wilc_p = stats.wilcoxon(scores_oud, scores_nieuw)
    print(f"👉 Wilcoxon test:          p-waarde = {wilc_p:.6f}  (stat = {wilc_stat:.1f})")

    # 3. Randomized Permutation Test (10.000 iteraties)
    waargenomen_verschil = np.abs(np.mean(scores_oud) - np.mean(scores_nieuw))
    samengevoegd = np.column_stack((scores_oud, scores_nieuw))
    num_queries = len(scores_oud)
    teller_extreem = 0

    np.random.seed(42) # Zorgt voor exact dezelfde resultaten bij herhaling
    for _ in range(10000):
        swap_mask = np.random.randint(0, 2, size=num_queries)
        perm_oud = np.where(swap_mask == 1, samengevoegd[:, 1], samengevoegd[:, 0])
        perm_nieuw = np.where(swap_mask == 1, samengevoegd[:, 0], samengevoegd[:, 1])
        if np.abs(np.mean(perm_oud) - np.mean(perm_nieuw)) >= waargenomen_verschil:
            teller_extreem += 1

    perm_p = teller_extreem / 10000
    print(f"👉 Permutation test:       p-waarde = {perm_p:.6f}")
    print("--------------------------------------------------")

    if perm_p < 0.05:
        print("🎉 CONCLUSIE: Het verschil is STATISTISCH SIGNIFICANT! (p < 0.05)")
    else:
        print("⚖️ CONCLUSIE: Het verschil is NIET statistisch significant. (p >= 0.05)")
    print("==================================================\n")

# Voer de testen uit voor beide metrieken!
voer_alle_testen_uit(df_query_scores, 'ndcg_10_old', 'ndcg_10_new', "NDCG@10")
voer_alle_testen_uit(df_query_scores, 'mrr_10_old', 'mrr_10_new', "MRR@10")